# 🧭 WanderAI — Fine-Tuning Llama 3.2 3B for Atmospheric Travel Narration
### Phase 5: LoRA Fine-Tuning with Unsloth (Colab Free T4 GPU)

This notebook trains a lightweight LoRA adapter on **Llama 3.2 3B Instruct** to generate evocative, sensory, cliché-free travel narrations for day-by-day itineraries, and exports a quantized GGUF model for zero-latency local serving via Ollama.

In [ ]:
# 1. Install Unsloth and training dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets transformers triton xformers

In [ ]:
# 2. Load Llama 3.2 3B Instruct with 4-bit Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None # None for auto detection (float16 on T4, bfloat16 on Ampere)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# 3. Add LoRA adapters (Rank = 16, Alpha = 32)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

In [ ]:
# 4. Format Dataset with Llama 3.2 ChatML Template
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are an expert travel writer specializing in evocative, atmospheric, "
    "and sensory travel narrations for day-by-day itineraries. "
    "Write exactly 1-2 sensory sentences (under 30 words) avoiding tourist clichés."
)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        user_content = f"{instruction}\n\nContext & Atmosphere: {input_text}" if input_text else instruction
        text = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            f"{SYSTEM_PROMPT}<|eot_id|>\n"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{user_content}<|eot_id|>\n"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{output}<|eot_id|>"
        )
        texts.append(text)
    return {"text": texts}

# Upload train.jsonl and eval.jsonl to Colab, then load:
dataset = load_dataset("json", data_files={"train": "train.jsonl", "eval": "eval.jsonl"})
train_dataset = dataset["train"].map(formatting_prompts_func, batched=True)
eval_dataset = dataset["eval"].map(formatting_prompts_func, batched=True)
print(f"Loaded {len(train_dataset)} training samples and {len(eval_dataset)} eval samples.")

In [ ]:
# 5. Train using SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.05,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        evaluation_strategy = "steps",
        eval_steps = 20,
        output_dir = "outputs",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
    ),
)

trainer_stats = trainer.train()
print("Training Complete!")

In [ ]:
# 6. Test Inference with Fine-Tuned Model
FastLanguageModel.for_inference(model)

test_prompt = "Write a 1-2 sentence evocative, atmospheric travel narration for Fontainhas Latin Quarter, a cultural in Goa.\n\nContext & Atmosphere: 18th-century Portuguese villas with pastel yellow walls and terracotta roofs."
inputs = tokenizer(
    [f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{test_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True, temperature = 0.4)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

In [ ]:
# 7. Save LoRA weights & Export GGUF for Ollama
model.save_pretrained("travel_narrator_lora")
tokenizer.save_pretrained("travel_narrator_lora")

# Export to 4-bit quantized GGUF for Ollama
model.save_pretrained_gguf("travel_narrator_gguf", tokenizer, quantization_method = "q4_k_m")
print("GGUF exported successfully! Download the file for Ollama.")